# 3. Fine-tune Clear Voice

Fine-tunes whisper-small on one fold's speakers and tests it on the other fold's speakers. **Run it twice: once with `HELDOUT = "A"`, then with `HELDOUT = "B"`.** About 60-90 minutes each on an L4. Keep this browser tab open while it trains.

In [ ]:
# Setup: run this cell first. On Colab it clones the repo, installs packages and mounts Google Drive.
REPO_URL = "https://github.com/armaan-khoja/clear-voice"
SMOKE = False  # True = a quick check on a handful of clips (minutes, not hours)

import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/clear-voice"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/clear-voice"], check=True)
    os.chdir("/content/clear-voice")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/clearvoice"
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret: downloads still work, uploading models will not.")
else:
    WORK = os.path.abspath("work")
sys.path.insert(0, os.getcwd())
os.makedirs(f"{WORK}/results", exist_ok=True)
os.makedirs(f"{WORK}/models", exist_ok=True)
SPLITS = "splits"
LIMIT = 6 if SMOKE else None
print("Results and models go to", WORK)

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "No GPU: Runtime > Change runtime type > L4 GPU, then rerun the setup cell.")

In [ ]:
HELDOUT = "A"        # run once with "A", then once with "B"
PUSH_TO_HUB = False  # True uploads the model to your Hugging Face account for the demo

In [ ]:
from clearvoice.pipeline import load_torgo
parquets, meta = load_torgo(WORK)  # downloads TORGO (about 1.6 GB) the first time in each session
print(len(meta), "clips from", meta.speaker.nunique(), "speakers")

In [ ]:
from clearvoice.data import load_audio
from clearvoice.pipeline import rows_for
name = f"hold{HELDOUT}"
train = rows_for(meta, SPLITS, name, "train")
val = rows_for(meta, SPLITS, name, "val")
if SMOKE:
    train, val = train.head(16), val.head(8)
print(len(train), "training clips from", sorted(train.speaker.unique()))
print(len(val), "validation clips (validation sentences only)")
audio = load_audio(parquets, meta, list(train.row_id) + list(val.row_id))

**Train.** Learning rate 1e-5, batch 16, up to 2,000 steps. Every 250 steps the model is scored on validation sentences it never trains on; the best one is kept, and training stops early if it stops improving.

In [ ]:
from clearvoice.train import TrainConfig, finetune
cfg = TrainConfig()
if SMOKE:
    cfg = TrainConfig(max_steps=4, eval_steps=2, save_steps=2, batch_size=4, eval_batch_size=4)
best = finetune(cfg, train, val, audio, f"{WORK}/models/{name}")
del audio
print("Kept model:", best)

**Test** on the held-out speakers (seen and unseen sentences), on the training speakers' unseen sentences, and on everyday English.

In [ ]:
import pandas as pd
from clearvoice.asr import load_model
from clearvoice.pipeline import librispeech_check, predict_fold
from clearvoice.report import leakage
model, processor = load_model(best)
preds = predict_fold(model, processor, parquets, meta, SPLITS, HELDOUT,
                     ["test_seen", "test_newsent", "test_strict"], "Clear Voice", batch_size=32, limit=LIMIT)
preds.to_csv(f"{WORK}/results/preds_ft_{name}.csv", index=False)
ls = librispeech_check(model, processor, n=20 if SMOKE else 200, batch_size=32)
pd.DataFrame([{"system": "Clear Voice", "fold": HELDOUT, "librispeech_wer": ls["wer"], "n": ls["n"]}]).to_csv(
    f"{WORK}/results/librispeech_ft_{name}.csv", index=False)
leakage(preds)

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import whoami
    repo = f"{whoami()['name']}/clear-voice-{name.lower()}"
    model.push_to_hub(repo)
    processor.push_to_hub(repo)
    print("Uploaded to https://huggingface.co/" + repo)